# Задание 2. Скалярный autograd

Нужен класс `Node` с тремя операциями: сложение, умножение и ReLU. Объекты скалярные. Производные считаются своим обратным проходом, без `torch.autograd` и без `torch.nn`. Числа остаются обычными Python-числами: для скаляров этого достаточно, а печать совпадает с примером из условия (`data=2`, а не `tensor(2.)`).

Узел хранит значение `data`, накопленный градиент `grad`, родителей `_prev` и локальную функцию `_backward`. Эта функция не возвращает градиент, а прибавляет его к родителям: один и тот же узел может войти в выражение несколько раз, и вклады обязаны сложиться.


## Локальные производные

Для выхода `out` с уже известным `out.grad`:

- `out = x + y`: оба родителя получают `out.grad`, потому что частные производные равны 1.
- `out = x * y`: `x` получает `y * out.grad`, `y` получает `x * out.grad`. Множители — это значения на прямом проходе, не градиенты.
- `out = relu(x)`: родитель получает `out.grad`, только если `x > 0`. В нуле берётся субградиент 0.

`backward` обходит граф от вызванного узла к листьям. Сначала топологическая сортировка (родители раньше детей в списке, затем список разворачивается), у стартового узла `grad = 1`, дальше каждый узел вызывает свой `_backward`. Повторный вызов градиенты прибавляет, как в micrograd: перед новым проходом их обнуляют снаружи.


## Пример из условия

В условии напечатано:

`Node(data=2, grad=0)  Node(data=-3, grad=10)  Node(data=10, grad=-3)  Node(data=-28, grad=1)  Node(data=0, grad=1)`

для `d = a + b * c`, `e = relu(d)`.

Значение `d = -28` верное: `2 + (-3) * 10`. Дальше цепное правило такое выражение не даёт. `e = relu(-28) = 0`, локальная производная ReLU на отрицательном аргументе равна 0, поэтому градиент в `d` и во все входы не идёт: у `a`, `b`, `c`, `d` градиент 0, у `e` градиент 1.

Строка из условия смешивает два разных режима. `d.grad = 1` означает, что ReLU градиент пропустил. Тогда у сложения `a.grad` тоже должен быть 1, а в строке он 0. `b.grad = 10` и `c.grad = -3` — это как раз производные произведения при `d.grad = 1`. Одновременно это не выполняется.

Ниже ReLU отрицательное число блокирует. Отдельный пример с положительным входом показывает ненулевые `b.grad` и `c.grad` той же формулой.


In [2]:
class Node:
    def __init__(self, data, _children=(), _op=""):
        self.data = data
        self.grad = 0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op

    def __repr__(self):
        return f"Node(data={self.data}, grad={self.grad})"

    def __add__(self, other):
        other = other if isinstance(other, Node) else Node(other)
        out = Node(self.data + other.data, (self, other), "+")

        def _backward():
            self.grad += out.grad
            other.grad += out.grad

        out._backward = _backward
        return out

    def __radd__(self, other):
        return self + other

    def __mul__(self, other):
        other = other if isinstance(other, Node) else Node(other)
        out = Node(self.data * other.data, (self, other), "*")

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad

        out._backward = _backward
        return out

    def __rmul__(self, other):
        return self * other

    def relu(self):
        out = Node(self.data if self.data > 0 else 0, (self,), "ReLU")

        def _backward():
            if self.data > 0:
                self.grad += out.grad

        out._backward = _backward
        return out

    def backward(self):
        order = []
        seen = set()

        def visit(node):
            if node in seen:
                return
            seen.add(node)
            for parent in node._prev:
                visit(parent)
            order.append(node)

        visit(self)
        self.grad = 1
        for node in reversed(order):
            node._backward()


In [3]:
a = Node(2)
b = Node(-3)
c = Node(10)
d = a + b * c
e = d.relu()
e.backward()
print(a, b, c, d, e)
print("d =", d.data, ", e =", e.data)


Node(data=2, grad=0) Node(data=-3, grad=0) Node(data=10, grad=0) Node(data=-28, grad=0) Node(data=0, grad=1)
d = -28 , e = 0


Тот же граф, но `c = -10`. Тогда `b * c = 30`, `d = 32 > 0`, ReLU действует как тождество, и градиенты доходят до входов: `a` получает 1, `b` получает значение `c` (`-10`), `c` получает значение `b` (`-3`).


In [4]:
a = Node(2)
b = Node(-3)
c = Node(-10)
d = a + b * c
e = d.relu()
e.backward()
print(a, b, c, d, e)


Node(data=2, grad=1) Node(data=-3, grad=-10) Node(data=-10, grad=-3) Node(data=32, grad=1) Node(data=32, grad=1)


## Тесты

`unittest` проверяет прямые значения, три локальные производные, общий узел в двух ветках (там градиент обязан сложиться из двух путей), ReLU в нуле и повторный `backward`. Отдельно конечные разности сверяют цепочку `(x * y + z).relu()` сразу по трём скалярам.


In [5]:
import unittest


class NodeTests(unittest.TestCase):
    def test_assignment_values_and_chain_rule(self):
        a = Node(2)
        b = Node(-3)
        c = Node(10)
        d = a + b * c
        e = d.relu()
        e.backward()
        self.assertEqual(d.data, -28)
        self.assertEqual(e.data, 0)
        self.assertEqual(e.grad, 1)
        self.assertEqual(d.grad, 0)
        self.assertEqual(a.grad, 0)
        self.assertEqual(b.grad, 0)
        self.assertEqual(c.grad, 0)

    def test_positive_relu_passes_product_rule(self):
        a = Node(2)
        b = Node(-3)
        c = Node(-10)
        d = a + b * c
        e = d.relu()
        e.backward()
        self.assertEqual(d.data, 32)
        self.assertEqual(e.data, 32)
        self.assertEqual((a.grad, b.grad, c.grad, d.grad, e.grad), (1, -10, -3, 1, 1))

    def test_add_and_mul_local(self):
        x = Node(4)
        y = Node(-5)
        added = x + y
        added.backward()
        self.assertEqual((added.data, x.grad, y.grad), (-1, 1, 1))

        x = Node(4)
        y = Node(-5)
        product = x * y
        product.backward()
        self.assertEqual((product.data, x.grad, y.grad), (-20, -5, 4))

    def test_shared_node_sums_both_paths(self):
        a = Node(2.0)
        b = Node(3.0)
        c = a * b + a
        c.backward()
        self.assertEqual(c.data, 8.0)
        self.assertEqual(a.grad, 4.0)
        self.assertEqual(b.grad, 2.0)

    def test_square_uses_the_same_node_twice(self):
        a = Node(5.0)
        square = a * a
        square.backward()
        self.assertEqual(square.data, 25.0)
        self.assertEqual(a.grad, 10.0)

    def test_relu_zero_has_zero_subgradient(self):
        a = Node(-1.5)
        out = a.relu()
        out.backward()
        self.assertEqual(out.data, 0)
        self.assertEqual(a.grad, 0)

        a = Node(0)
        out = a.relu()
        out.backward()
        self.assertEqual(out.data, 0)
        self.assertEqual(a.grad, 0)

        a = Node(1.5)
        out = a.relu()
        out.backward()
        self.assertEqual(out.data, 1.5)
        self.assertEqual(a.grad, 1)

    def test_scalar_operands(self):
        a = Node(3)
        out = 2 * (a + 4)
        out.backward()
        self.assertEqual(out.data, 14)
        self.assertEqual(a.grad, 2)

    def test_second_backward_accumulates(self):
        a = Node(2.0)
        b = a * Node(3.0)
        b.backward()
        b.backward()
        self.assertEqual(a.grad, 6.0)

    def test_finite_difference(self):
        def value_at(x_value, y_value, z_value):
            output = (Node(x_value) * Node(y_value) + Node(z_value)).relu()
            return output.data

        x = Node(1.5)
        y = Node(-2.0)
        z = Node(4.0)
        output = (x * y + z).relu()
        output.backward()
        step = 1e-6
        expected = {
            x: (value_at(1.5 + step, -2.0, 4.0) - value_at(1.5 - step, -2.0, 4.0)) / (2 * step),
            y: (value_at(1.5, -2.0 + step, 4.0) - value_at(1.5, -2.0 - step, 4.0)) / (2 * step),
            z: (value_at(1.5, -2.0, 4.0 + step) - value_at(1.5, -2.0, 4.0 - step)) / (2 * step),
        }
        for node, derivative in expected.items():
            self.assertAlmostEqual(node.grad, derivative, places=5)


suite = unittest.defaultTestLoader.loadTestsFromTestCase(NodeTests)
result = unittest.TextTestRunner(verbosity=2).run(suite)
assert result.wasSuccessful()


test_add_and_mul_local (__main__.NodeTests.test_add_and_mul_local) ... ok
test_assignment_values_and_chain_rule (__main__.NodeTests.test_assignment_values_and_chain_rule) ... ok
test_finite_difference (__main__.NodeTests.test_finite_difference) ... ok
test_positive_relu_passes_product_rule (__main__.NodeTests.test_positive_relu_passes_product_rule) ... ok
test_relu_zero_has_zero_subgradient (__main__.NodeTests.test_relu_zero_has_zero_subgradient) ... ok
test_scalar_operands (__main__.NodeTests.test_scalar_operands) ... ok
test_second_backward_accumulates (__main__.NodeTests.test_second_backward_accumulates) ... ok
test_shared_node_sums_both_paths (__main__.NodeTests.test_shared_node_sums_both_paths) ... ok
test_square_uses_the_same_node_twice (__main__.NodeTests.test_square_uses_the_same_node_twice) ... ok

----------------------------------------------------------------------
Ran 9 tests in 0.006s

OK
